# 00 - Préparation des données

**Objectif du projet** : prédire le comportement d'un chien (couché, assis, marche, renifle...) à partir de
séries temporelles de capteurs de mouvement (accéléromètres et gyroscopes placés sur le dos et sur le cou).

**Données** : 45 chiens, 62 séances (`DogID` × `TestNum`), signaux échantillonnés à 100 Hz, comportements
annotés à partir de vidéos (`Behavior_1`, `Behavior_2`, `Behavior_3` : comportements simultanés).
Ce jeu de données accompagne l'article de Kumpulainen et al. (2021), *Dog behaviour classification with
movement sensors placed on the harness and the collar*, Applied Animal Behaviour Science
(référence à vérifier et à citer dans le rendu).

**Ce notebook** :
1. lit le CSV brut une seule fois et l'enregistre en parquet (types compacts) : les notebooks suivants
   rechargent les données en quelques secondes ;
2. traite les étiquettes techniques de synchronisation ;
3. contrôle la continuité temporelle des signaux et la couverture des annotations.

**Pré-requis** : placer `DogMoveData.csv` et `DogInfo.xlsx` dans `data/raw/`.

In [3]:
import sys
from pathlib import Path

# Racine du projet ajoutée au chemin d'import (le notebook est dans notebooks/)
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from src import config as cfg
pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

from src import data

## 1. Construction du cache

Parti pris sur la **synchronisation** : les lignes étiquetées `Synchronization` / `Extra_Synchronization`
servent à caler les capteurs sur la vidéo ; ce ne sont pas des comportements. On **conserve les lignes**
(le signal reste continu) mais leur label devient manquant. La première version supprimait ces lignes, ce
qui créait des trous dans le signal que le fenêtrage enjambait sans le savoir.

In [4]:
data.build_cache(force=False)   # force=True pour reconstruire après modification des fichiers bruts
moves = data.load_moves()
dogs = data.load_dogs()
print(f"Séries temporelles : {moves.shape[0]:,} lignes x {moves.shape[1]} colonnes")
print(f"Chiens : {len(dogs)} | séances : {moves.groupby(cfg.SESSION_KEYS).ngroups}")
print(f"Mémoire occupée : {moves.memory_usage(deep=True).sum() / 1e9:.2f} Go")
display(moves.head())

Lecture du CSV brut (peut prendre une minute)...
Cache écrit : 10,611,068 lignes de mouvement, 45 chiens.
Séries temporelles : 10,611,068 lignes x 20 colonnes
Chiens : 45 | séances : 62
Mémoire occupée : 0.70 Go


,DogID,TestNum,t_sec,ABack_x,ABack_y,ABack_z,ANeck_x,ANeck_y,ANeck_z,GBack_x,GBack_y,GBack_z,GNeck_x,GNeck_y,GNeck_z,Task,Behavior_1,Behavior_2,Behavior_3,PointEvent
0,16,1,0.00,0.041504,0.938965,-0.015137,-0.067871,-0.510254,-0.934570,-17.639160,-22.766115,7.446290,-7.934571,6.347657,13.427735,NaN,NaN,NaN,NaN,NaN
1,16,1,0.01,0.041992,0.941895,-0.020020,-0.128906,-0.494141,-0.913086,-15.075685,-11.413575,4.821778,-3.906250,4.394532,16.540527,NaN,NaN,NaN,NaN,NaN
2,16,1,0.02,0.040527,0.939453,-0.004395,-0.158691,-0.480469,-0.911133,-12.207032,-0.122070,2.807617,-0.488281,-1.953125,26.794436,NaN,NaN,NaN,NaN,NaN
3,16,1,0.03,0.021484,0.946289,0.007813,-0.122070,-0.486816,-0.880371,-9.460450,7.995606,1.586914,1.159668,-5.676270,38.085941,NaN,NaN,NaN,NaN,NaN
4,16,1,0.04,-0.000977,0.951172,0.033691,-0.053711,-0.500000,-0.807129,-8.361817,14.587403,-1.037598,4.577637,4.089356,41.503910,NaN,NaN,NaN,NaN,NaN


## 2. Continuité temporelle

Une fenêtre d'analyse ne doit jamais enjamber une rupture (saut de `t_sec`). Le fenêtrage découpe donc
chaque séance en **segments continus**. Ce tableau indique combien de ruptures existent.

In [5]:
continuity = data.continuity_report(moves)
print(f"Séances avec au moins une rupture : {(continuity['n_ruptures'] > 0).sum()} / {len(continuity)}")
display(continuity.sort_values("n_ruptures", ascending=False).head(10))

Séances avec au moins une rupture : 0 / 62


,DogID,TestNum,n_lignes,duree_min,n_ruptures,plus_grand_trou_s
0,16,1,164004,27.33,0,0.01
1,16,2,156872,26.15,0,0.01
2,18,1,175336,29.22,0,0.01
3,19,2,157376,26.23,0,0.01
4,20,1,185904,30.98,0,0.01
5,21,1,173332,28.89,0,0.01
6,22,2,160404,26.73,0,0.01
7,23,1,168108,28.02,0,0.01
8,23,2,151028,25.17,0,0.01
9,25,1,164512,27.42,0,0.01


## 3. Couverture des annotations

Une partie importante du temps n'est pas annotée (transitions entre tâches, chien hors champ de la caméra...).
**Parti pris** : la tâche porte sur la classification des segments annotés. Les fenêtres non annotées sont
exclues de l'apprentissage et de l'évaluation (voir notebook 03).

In [6]:
coverage = data.label_coverage(moves)
display(coverage.describe().loc[["mean", "min", "max"]].round(1))
missing = (moves[cfg.BEHAVIOR_COLUMNS + ["Task"]].isna().mean() * 100).round(1)
print("Pourcentage de lignes sans annotation :")
print(missing.to_string())

,DogID,TestNum,n_lignes,duree_min,couverture_B1_%,couverture_B2_%,couverture_B3_%
mean,43.3,1.4,171146.3,28.5,62.3,28.2,1.9
min,16.0,1.0,146584.0,24.4,45.7,9.9,0.0
max,74.0,2.0,217656.0,36.3,82.5,43.0,5.4


Pourcentage de lignes sans annotation :
Behavior_1    38.2
Behavior_2    71.9
Behavior_3    98.1
Task          24.4
